In [3]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# 1. Set your local PlantVillage folder path
plantvillage_raw_path = Path("/Users/kushrocks6306/Deep Learning Project/PlantVillage- db")

# Locate the 'color' subfolder automatically so grayscale and segmented folders are skipped
if (plantvillage_raw_path / "plantvillage dataset" / "color").exists():
    pv_color_path = plantvillage_raw_path / "plantvillage dataset" / "color"
elif (plantvillage_raw_path / "color").exists():
    pv_color_path = plantvillage_raw_path / "color"
elif plantvillage_raw_path.exists():
    pv_color_path = plantvillage_raw_path
else:
    raise FileNotFoundError(f"PlantVillage folder not found at: {plantvillage_raw_path}")

print(f"Reading PlantVillage color images from: {pv_color_path}")

# 2. Output folder for 128x128 training set
pv_output_path = Path("plantvillage_train_128x128")
pv_output_path.mkdir(parents=True, exist_ok=True)

def compute_dhash(img, hash_size=8):
    img_gray = img.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
    pixels = np.asarray(img_gray, dtype=np.int16)
    return (pixels[:, 1:] > pixels[:, :-1]).flatten()

similarity_threshold = 0.80
max_diff_bits = int(64 * (1.0 - similarity_threshold))
valid_extensions = {".jpg", ".jpeg", ".png"}

pv_class_counts = {}
total_scanned = 0
total_saved = 0
duplicates_skipped = 0
errors = 0

for folder in sorted(pv_color_path.iterdir()):
    if not folder.is_dir():
        continue

    label = folder.name
    save_folder = pv_output_path / label
    save_folder.mkdir(parents=True, exist_ok=True)

    seen_hashes = []
    folder_saved_count = 0

    for file in sorted(folder.iterdir()):
        if file.suffix.lower() not in valid_extensions:
            continue

        total_scanned += 1

        try:
            with Image.open(file) as img:
                img_rgb = img.convert("RGB")
                current_hash = compute_dhash(img_rgb)

                # Check for duplicates within the class
                is_duplicate = False
                if seen_hashes:
                    seen_matrix = np.array(seen_hashes)
                    diff_counts = np.count_nonzero(seen_matrix != current_hash, axis=1)
                    if np.any(diff_counts <= max_diff_bits):
                        is_duplicate = True

                if is_duplicate:
                    duplicates_skipped += 1
                    continue

                seen_hashes.append(current_hash)
                save_path = save_folder / f"{file.stem}.jpg"

                if not save_path.exists():
                    img_resized = img_rgb.resize((128, 128), Image.Resampling.LANCZOS)
                    img_resized.save(save_path, "JPEG", quality=95)

                folder_saved_count += 1
                total_saved += 1

        except Exception as e:
            errors += 1
            print(f"Error processing {file.name}: {e}")

    pv_class_counts[label] = folder_saved_count
    print(f"[PlantVillage] {label}: {folder_saved_count} unique images saved")

print("\n--- PlantVillage Training Set Complete ---")
print(f"Total classes: {len(pv_class_counts)}")
print(f"Total scanned: {total_scanned} | Unique saved: {total_saved} | Duplicates skipped: {duplicates_skipped} | Errors: {errors}")
print(f"Training dataset ready at: {pv_output_path.resolve()}") 

Reading PlantVillage color images from: /Users/kushrocks6306/Deep Learning Project/PlantVillage- db
[PlantVillage] Pepper__bell___Bacterial_spot: 870 unique images saved
[PlantVillage] Pepper__bell___healthy: 922 unique images saved
[PlantVillage] PlantVillage: 0 unique images saved
[PlantVillage] Potato___Early_blight: 876 unique images saved
[PlantVillage] Potato___Late_blight: 790 unique images saved
[PlantVillage] Potato___healthy: 128 unique images saved
[PlantVillage] Tomato_Bacterial_spot: 1202 unique images saved
[PlantVillage] Tomato_Early_blight: 886 unique images saved
[PlantVillage] Tomato_Late_blight: 1542 unique images saved
[PlantVillage] Tomato_Leaf_Mold: 743 unique images saved
[PlantVillage] Tomato_Septoria_leaf_spot: 1477 unique images saved
[PlantVillage] Tomato_Spider_mites_Two_spotted_spider_mite: 867 unique images saved
[PlantVillage] Tomato__Target_Spot: 1029 unique images saved
[PlantVillage] Tomato__Tomato_YellowLeaf__Curl_Virus: 2334 unique images saved
[Plan

In [5]:
pip install datasets huggingface_hub pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 16.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 12.5 MB/s  0:00:000.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 7.1 MB/s  0:00:05 eta 0:00:010:00:01m
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.05;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/16 [typing-extensions]
  Attempting uninstall: tqdm0m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/16 [truststore]xtensions]
    Found existing installation: tqdm 4.66.138;5;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/16 [truststore]
    Uninstalling tqdm-4.66.1:8;5;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/16 [truststore]
      Successfully uninstalled tqdm-4.66.1━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/16 [truststor

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
from huggingface_hub import login
login(token="YOUR_HUGGINGFACE_TOKEN")

In [4]:
import resource
import os

# Increase maximum open file descriptor limit on macOS/Linux
soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
resource.setrlimit(resource.RLIMIT_NOFILE, (hard, hard))

from datasets import load_dataset

# Hugging Face will automatically detect cached/partially downloaded files
ds = load_dataset("lorenzoxi/tomato-leaves-dataset")

Resolving data files:   0%|          | 0/14404 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1796 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1796 [00:00<?, ?it/s]

dataset-tomatoes/train/Tomato___Tomato_m(…): reconstructing file:   0%|          |  0.00B / 16.6kB            

dataset-tomatoes/train/Tomato___Tomato_m(…): reconstructing file:   0%|          |  0.00B / 18.2kB            

dataset-tomatoes/train/Tomato___Tomato_m(…): reconstructing file:   0%|          |  0.00B / 14.4kB            

... (10,646 outputs left out)

dataset-tomatoes/test/Tomato___healthy/t(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14404 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1796 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1796 [00:00<?, ? examples/s]

In [5]:
from datasets import load_dataset

# Replace this with the actual path to your project folder
project_folder = '/Users/kushrocks6306/Deep Learning Project/Proccessed Dataset'

ds = load_dataset("lorenzoxi/tomato-leaves-dataset", cache_dir=project_folder)

Resolving data files:   0%|          | 0/14404 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1796 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1796 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/14404 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1796 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1796 [00:00<?, ? examples/s]

In [10]:
import os
from datasets import load_dataset

# Load the dataset (uses your existing downloaded files)
ds = load_dataset("lorenzoxi/tomato-leaves-dataset")

output_dir = "/Users/kushrocks6306/Deep Learning Project/Dataset/Images"
print("Starting image extraction...")

for split in ds.keys():
    split_dir = os.path.join(output_dir, split)
    os.makedirs(split_dir, exist_ok=True)
    
    for i, item in enumerate(ds[split]):
        img = item["image"]
        label = item["label"]
        
        # Convert RGBA (transparent) or other formats to standard RGB for JPEG
        if img.mode != "RGB":
            img = img.convert("RGB")
            
        # Save as a standard JPG
        file_path = os.path.join(split_dir, f"tomato_{label}_{i}.jpg")
        img.save(file_path)

print(f"✅ Done! All images have been extracted to: {output_dir}") 

Resolving data files:   0%|          | 0/14404 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1796 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1796 [00:00<?, ?it/s]

Starting image extraction...
✅ Done! All images have been extracted to: /Users/kushrocks6306/Deep Learning Project/Dataset/Images


In [11]:
import os
from datasets import load_dataset

# Load the dataset
ds = load_dataset("lorenzoxi/tomato-leaves-dataset")

# Extract the actual string names of the classes from the dataset metadata
class_names = ds['train'].features['label'].names

output_dir = "/Users/kushrocks6306/Deep Learning Project/Dataset/Images"
print("Starting classified image extraction...")

for split in ds.keys():
    for i, item in enumerate(ds[split]):
        img = item["image"]
        label_idx = item["label"]
        
        # Get the actual disease name (e.g., 'Bacterial_spot', 'healthy')
        disease_name = class_names[label_idx]
        
        # Create the subfolder path: Dataset/Images/train/Bacterial_spot/
        class_dir = os.path.join(output_dir, split, disease_name)
        os.makedirs(class_dir, exist_ok=True)
        
        # Convert RGBA (transparent) to standard RGB for JPEG
        if img.mode != "RGB":
            img = img.convert("RGB")
            
        # Save the image directly into its specific disease folder
        file_path = os.path.join(class_dir, f"tomato_{disease_name}_{i}.jpg")
        img.save(file_path)

print(f"✅ Done! All images are now classified into their specific folders in: {output_dir}")

Resolving data files:   0%|          | 0/14404 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1796 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1796 [00:00<?, ?it/s]

Starting classified image extraction...
✅ Done! All images are now classified into their specific folders in: /Users/kushrocks6306/Deep Learning Project/Dataset/Images


In [13]:
import numpy as np
from pathlib import Path
from PIL import Image

# 1. Direct path to the single category folder
target_folder = Path("/Users/kushrocks6306/Deep Learning Project/Original Dataset/PlantVillage- db/Tomato_Bacterial_spot")

if not target_folder.exists():
    raise FileNotFoundError(f"Folder not found at: {target_folder}")

print(f"Reading images directly from: {target_folder}")

# 2. Output folder specifically for Tomato_Bacterial_spot
pv_output_path = Path("/Users/kushrocks6306/Deep Learning Project/Proccessed Dataset/plantvillage_train_128x128/Tomato_Bacterial_spot")
pv_output_path.mkdir(parents=True, exist_ok=True)

def compute_dhash(img, hash_size=8):
    img_gray = img.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
    pixels = np.asarray(img_gray, dtype=np.int16)
    return (pixels[:, 1:] > pixels[:, :-1]).flatten()

similarity_threshold = 0.80
max_diff_bits = int(64 * (1.0 - similarity_threshold))
valid_extensions = {".jpg", ".jpeg", ".png"}

total_scanned = 0
total_saved = 0
duplicates_skipped = 0
errors = 0

seen_hashes = []

# Iterate directly through image files in the target folder
for file in sorted(target_folder.iterdir()):
    if not file.is_file() or file.suffix.lower() not in valid_extensions:
        continue

    total_scanned += 1

    try:
        with Image.open(file) as img:
            img_rgb = img.convert("RGB")
            current_hash = compute_dhash(img_rgb)

            # Check for duplicates
            is_duplicate = False
            if seen_hashes:
                seen_matrix = np.array(seen_hashes)
                diff_counts = np.count_nonzero(seen_matrix != current_hash, axis=1)
                if np.any(diff_counts <= max_diff_bits):
                    is_duplicate = True

            if is_duplicate:
                duplicates_skipped += 1
                continue

            seen_hashes.append(current_hash)
            save_path = pv_output_path / f"{file.stem}.jpg"

            # Always save or overwrite to restore missing/deleted files
            img_resized = img_rgb.resize((128, 128), Image.Resampling.LANCZOS)
            img_resized.save(save_path, "JPEG", quality=95)

            total_saved += 1

    except Exception as e:
        errors += 1
        print(f"Error processing {file.name}: {e}")

print("\n--- Tomato_Bacterial_spot Processing Complete ---")
print(f"Total scanned: {total_scanned} | Unique saved: {total_saved} | Duplicates skipped: {duplicates_skipped} | Errors: {errors}")
print(f"Restored dataset saved at: {pv_output_path.resolve()}")

Reading images directly from: /Users/kushrocks6306/Deep Learning Project/Original Dataset/PlantVillage- db/Tomato_Bacterial_spot

--- Tomato_Bacterial_spot Processing Complete ---
Total scanned: 2127 | Unique saved: 1202 | Duplicates skipped: 925 | Errors: 0
Restored dataset saved at: /Users/kushrocks6306/Deep Learning Project/Proccessed Dataset/plantvillage_train_128x128/Tomato_Bacterial_spot
